# JRN-Guided Heterogeneous Architecture: GBM Probe Estimation on rel-stack

This notebook demonstrates the **Join Reproduction Number (JRN)** methodology for evaluating
FK join informativeness in relational learning. The experiment:

1. **Computes GBM-based JRN** for all reachable FK joins across 3 rel-stack tasks
   (user-engagement, user-badge, post-votes) using LightGBM probes with 3 seeds.
2. **Classifies joins** as HIGH/CRITICAL/LOW based on JRN thresholds.
3. **Compares 5 architecture configurations**: JRN-guided, uniform-mean, uniform-rich, top-K, oracle (greedy forward selection).
4. **Analyzes results**: win rates, oracle gaps, aggregation sensitivity, and feature efficiency.

Since the full experiment requires downloading the rel-stack dataset and training many LightGBM models,
this demo loads **pre-computed results** and demonstrates the analysis and visualization pipeline.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0', 'tabulate==0.9.0')

In [ ]:
import json
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from tabulate import tabulate

warnings.filterwarnings("ignore")
print("Imports OK")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-bc07ab-join-reproduction-number-epidemiology-in/main/experiment_iter4_jrn_guided_hete/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded data with keys: {list(data.keys())}")
print(f"  metadata keys: {list(data['metadata'].keys())}")
print(f"  datasets: {len(data['datasets'])} dataset(s), "
      f"{len(data['datasets'][0]['examples'])} examples")

## Configuration

These are the tunable parameters from the original experiment. The pre-computed results
used these values; we define them here for reference and for re-deriving analysis metrics.

In [ ]:
# ============================================================
# Configuration — tunable parameters from the original experiment
# ============================================================

# LightGBM probe parameters (used for JRN estimation)
PROBE_PARAMS = {
    "n_estimators": 150,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "max_bin": 127,
}

# LightGBM final evaluation parameters
FINAL_PARAMS = {
    "n_estimators": 250,
    "max_depth": 8,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "max_bin": 255,
}

SEEDS = [42, 123, 456]
MAX_TRAIN_SAMPLES = 150_000
MAX_VAL_SAMPLES = 50_000
MAX_TEST_SAMPLES = 50_000
AGG_TYPES = ["mean", "sum", "max", "std"]

# JRN classification thresholds
JRN_HIGH = 1.15
JRN_LOW = 0.85

print(f"Config loaded: {len(SEEDS)} seeds, JRN thresholds: HIGH>{JRN_HIGH}, LOW<{JRN_LOW}")

## Constants & Schema

The rel-stack dataset schema defines tables, primary keys, and FK relationships.
Each FK join is a candidate for JRN estimation.

In [ ]:
# ============================================================
# Constants & Schema (from original method.py)
# ============================================================

SCHEMA = {
    "users":       {"pkey": "Id", "time_col": "CreationDate", "fkeys": {}},
    "posts":       {"pkey": "Id", "time_col": "CreationDate",
                    "fkeys": {"OwnerUserId": "users", "ParentId": "posts"}},
    "comments":    {"pkey": "Id", "time_col": "CreationDate",
                    "fkeys": {"UserId": "users", "PostId": "posts"}},
    "votes":       {"pkey": "Id", "time_col": "CreationDate",
                    "fkeys": {"PostId": "posts", "UserId": "users"}},
    "badges":      {"pkey": "Id", "time_col": "Date",
                    "fkeys": {"UserId": "users"}},
    "postLinks":   {"pkey": "Id", "time_col": "CreationDate",
                    "fkeys": {"PostId": "posts", "RelatedPostId": "posts"}},
    "postHistory": {"pkey": "Id", "time_col": "CreationDate",
                    "fkeys": {"PostId": "posts", "UserId": "users"}},
}

ALL_JOINS = [
    ("comments",    "UserId",        "users"),
    ("comments",    "PostId",        "posts"),
    ("badges",      "UserId",        "users"),
    ("postLinks",   "PostId",        "posts"),
    ("postLinks",   "RelatedPostId", "posts"),
    ("postHistory", "PostId",        "posts"),
    ("postHistory", "UserId",        "users"),
    ("votes",       "PostId",        "posts"),
    ("votes",       "UserId",        "users"),
    ("posts",       "OwnerUserId",   "users"),
    ("posts",       "ParentId",      "posts"),
]

TASKS = {
    "user-engagement": {
        "entity_table": "users", "entity_col": "OwnerUserId",
        "target_col": "contribution", "task_type": "classification",
        "metric": "average_precision",
    },
    "user-badge": {
        "entity_table": "users", "entity_col": "UserId",
        "target_col": "WillGetBadge", "task_type": "classification",
        "metric": "average_precision",
    },
    "post-votes": {
        "entity_table": "posts", "entity_col": "PostId",
        "target_col": "popularity", "task_type": "regression",
        "metric": "neg_mae",
    },
}

def _jk(child: str, fk: str, parent: str) -> str:
    return f"{child}.{fk} -> {parent}"

print(f"Schema: {len(SCHEMA)} tables, {len(ALL_JOINS)} joins, {len(TASKS)} tasks")

## Extract JRN Matrix

The JRN matrix shows the Join Reproduction Number for each (join, task) pair.
JRN > 1.15 means the join is **highly informative** (HIGH), JRN in [0.85, 1.15] is **critical**
(marginal benefit), and JRN < 0.85 means the join is **uninformative** (LOW).

In [ ]:
# ============================================================
# Extract JRN Matrix from pre-computed results
# ============================================================

meta = data["metadata"]
jrn_matrix = meta["jrn_matrix"]
joins = jrn_matrix["joins"]
tasks = jrn_matrix["tasks"]
values = jrn_matrix["values"]
categories = jrn_matrix["categories"]

# Build a pandas DataFrame for display
jrn_df = pd.DataFrame(values, index=joins, columns=tasks)
cat_df = pd.DataFrame(categories, index=joins, columns=tasks)

print("JRN Matrix (values):")
print(jrn_df.round(3).to_string())
print()
print("JRN Categories:")
print(cat_df.to_string())

## Baseline & Configuration Results

Compare 5 architecture configurations per task:
- **jrn_guided**: Uses JRN categories to select joins (HIGH -> mean agg, CRITICAL -> all aggs)
- **uniform_mean**: All joins with mean aggregation
- **uniform_rich**: All joins with all aggregations
- **top_k**: Top-K joins ranked by JRN value
- **oracle**: Greedy forward selection (best possible)

In [ ]:
# ============================================================
# Extract baseline & config results from pre-computed data
# ============================================================

baseline_scores = meta["baseline_scores"]
config_results = meta["config_results"]

configs = ["jrn_guided", "uniform_mean", "uniform_rich", "top_k", "oracle"]

for tn in tasks:
    print(f"\n{'='*60}")
    print(f"Task: {tn}  (metric: {TASKS[tn]['metric']})")
    print(f"{'='*60}")
    bl = baseline_scores[tn]
    print(f"  Baseline (entity-only): {bl['mean']:.6f} +/- {bl['std']:.6f}")
    print()
    rows = []
    for cfg in configs:
        cr = config_results[tn][cfg]
        rows.append([cfg, f"{cr['mean']:.6f}", f"{cr['std']:.6f}",
                      cr['n_features'], len(cr['joins_used'])])
    print(tabulate(rows, headers=["Config", "Mean Score", "Std", "# Features", "# Joins"],
                   tablefmt="simple"))

## Analysis: Win Rates, Oracle Gap & Feature Efficiency

Re-derive key analysis metrics from the pre-computed results, mirroring
the `compute_analysis()` function from the original script.

In [ ]:
# ============================================================
# Re-derive analysis metrics (from original compute_analysis)
# ============================================================

# --- Win rates: JRN-guided vs each alternative ---
print("Win Rates (JRN-guided vs alternatives):")
print("-" * 50)
for other in ["uniform_mean", "uniform_rich", "top_k"]:
    w, t, l = 0, 0, 0
    for tn in tasks:
        jg = config_results[tn]["jrn_guided"]["mean"]
        ot = config_results[tn][other]["mean"]
        if jg > ot + 1e-4:
            w += 1
        elif ot > jg + 1e-4:
            l += 1
        else:
            t += 1
    print(f"  jrn_guided vs {other:15s}: wins={w}, ties={t}, losses={l}, "
          f"rate={w/len(tasks):.2f}")

# --- Oracle gap: how close is JRN-guided to oracle? ---
print(f"\nOracle Gap Analysis:")
print("-" * 50)
oracle_gaps = {}
for tn in tasks:
    o = config_results[tn]["oracle"]["mean"]
    j = config_results[tn]["jrn_guided"]["mean"]
    b = baseline_scores[tn]["mean"]
    gap = (o - j) / (o - b) if (o - b) != 0 else 0.0
    oracle_gaps[tn] = {"gap": round(gap, 4), "oracle": o,
                        "jrn_guided": j, "baseline": b}
    print(f"  {tn:20s}: gap={gap:.2%}  (oracle={o:.6f}, jrn={j:.6f}, base={b:.6f})")

# --- Feature efficiency ---
print(f"\nFeature Efficiency (gain per feature):")
print("-" * 50)
fe_rows = []
for tn in tasks:
    b_m = baseline_scores[tn]["mean"]
    for cfg in configs:
        cr = config_results[tn][cfg]
        gain = cr["mean"] - b_m
        nf = cr["n_features"]
        eff = gain / nf if nf else 0
        fe_rows.append([tn, cfg, nf, f"{gain:.6f}", f"{eff:.6f}"])
print(tabulate(fe_rows,
               headers=["Task", "Config", "# Feat", "Gain", "Efficiency"],
               tablefmt="simple"))

## Visualization

Four key plots:
1. **JRN Heatmap** — shows join informativeness across tasks
2. **Config Performance Comparison** — bar chart comparing all 5 configurations per task
3. **Oracle Gap** — how close JRN-guided is to the oracle upper bound
4. **Feature Efficiency** — gain per feature for each configuration

In [ ]:
# ============================================================
# Visualization
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# --- Plot 1: JRN Heatmap ---
ax = axes[0, 0]
# Replace None with NaN for plotting
heatmap_data = jrn_df.astype(float).values
mask = np.isnan(heatmap_data)

# Custom colormap
cmap = plt.cm.RdYlGn
norm = mcolors.Normalize(vmin=0.8, vmax=4.5)

im = ax.imshow(np.where(mask, np.nan, heatmap_data), cmap=cmap, norm=norm,
               aspect='auto', interpolation='nearest')
ax.set_xticks(range(len(tasks)))
ax.set_xticklabels([t.replace('-', '\n') for t in tasks], fontsize=9)
ax.set_yticks(range(len(joins)))
ax.set_yticklabels([j.replace(' -> ', '\n-> ') for j in joins], fontsize=7)

# Annotate cells
for i in range(len(joins)):
    for j_idx in range(len(tasks)):
        val = heatmap_data[i, j_idx]
        cat = categories[i][j_idx]
        if not np.isnan(val):
            color = 'white' if val > 3 else 'black'
            ax.text(j_idx, i, f"{val:.2f}\n{cat}",
                    ha='center', va='center', fontsize=7, color=color,
                    fontweight='bold')
        else:
            ax.text(j_idx, i, "N/A", ha='center', va='center',
                    fontsize=7, color='gray')

plt.colorbar(im, ax=ax, label='JRN Value', shrink=0.8)
ax.set_title('JRN Matrix: Join Informativeness per Task', fontsize=11, fontweight='bold')

# --- Plot 2: Config Performance Comparison ---
ax = axes[0, 1]
x = np.arange(len(tasks))
width = 0.15
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']

for i, cfg in enumerate(configs):
    means = [config_results[tn][cfg]["mean"] for tn in tasks]
    stds = [config_results[tn][cfg]["std"] for tn in tasks]
    ax.bar(x + i * width, means, width, label=cfg.replace('_', ' '),
           color=colors[i], yerr=stds, capsize=2, alpha=0.85)

# Add baseline as dashed lines
for j_idx, tn in enumerate(tasks):
    bl_val = baseline_scores[tn]["mean"]
    ax.hlines(bl_val, j_idx - 0.1, j_idx + len(configs) * width,
              colors='red', linestyles='dashed', linewidth=1, alpha=0.7)

ax.set_xticks(x + width * 2)
ax.set_xticklabels(tasks, fontsize=9)
ax.set_ylabel('Score (higher is better)', fontsize=9)
ax.legend(fontsize=7, loc='upper left')
ax.set_title('Config Performance Comparison', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# --- Plot 3: Oracle Gap ---
ax = axes[1, 0]
gap_tasks = list(oracle_gaps.keys())
gap_vals = [oracle_gaps[tn]["gap"] * 100 for tn in gap_tasks]
bar_colors = ['#66BB6A' if g < 5 else '#FFA726' if g < 10 else '#EF5350'
              for g in gap_vals]
bars = ax.bar(gap_tasks, gap_vals, color=bar_colors, edgecolor='black', alpha=0.85)
ax.set_ylabel('Oracle Gap (%)', fontsize=10)
ax.set_title('Oracle Gap: JRN-Guided vs Oracle (lower is better)', fontsize=11,
             fontweight='bold')
ax.axhline(y=5, color='green', linestyle='--', alpha=0.5, label='5% threshold')
ax.axhline(y=10, color='orange', linestyle='--', alpha=0.5, label='10% threshold')
ax.legend(fontsize=8)
for bar, val in zip(bars, gap_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# --- Plot 4: Feature Efficiency ---
ax = axes[1, 1]
eff_data = {}
for tn in tasks:
    b_m = baseline_scores[tn]["mean"]
    for cfg in configs:
        cr = config_results[tn][cfg]
        gain = cr["mean"] - b_m
        nf = cr["n_features"]
        eff = (gain / nf * 1000) if nf else 0  # x1000 for readability
        if cfg not in eff_data:
            eff_data[cfg] = []
        eff_data[cfg].append(eff)

x = np.arange(len(tasks))
for i, cfg in enumerate(configs):
    ax.bar(x + i * width, eff_data[cfg], width, label=cfg.replace('_', ' '),
           color=colors[i], alpha=0.85)

ax.set_xticks(x + width * 2)
ax.set_xticklabels(tasks, fontsize=9)
ax.set_ylabel('Efficiency (gain/feature x1000)', fontsize=9)
ax.legend(fontsize=7, loc='upper left')
ax.set_title('Feature Efficiency by Config', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('jrn_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved jrn_analysis.png")

## Domain Validation & Aggregation Sensitivity

Verify that domain-expected joins are correctly identified as informative,
and examine how sensitive JRN estimates are to aggregation type choice.

In [ ]:
# ============================================================
# Domain Validation & Aggregation Sensitivity (from original)
# ============================================================

analysis = meta["analysis"]

# Domain validation
print("Domain Validation:")
print("-" * 60)
dv = analysis["domain_validation"]
for tn, checks in dv.items():
    for check in checks:
        status = check.get("passed", "N/A")
        jrn_val = check.get("jrn", "N/A")
        actual = check.get("actual", "N/A")
        print(f"  {tn:20s} | {check['join']:30s} | "
              f"actual={actual:8s} | JRN={jrn_val}")

# Aggregation sensitivity
print(f"\nAggregation Sensitivity (top 10 by sensitivity):")
print("-" * 60)
ag_sens = analysis["aggregation_sensitivity"]
sorted_sens = sorted(ag_sens.items(), key=lambda x: x[1]["sensitivity"], reverse=True)
rows = []
for key, val in sorted_sens[:10]:
    rows.append([key, f"{val['jrn']:.3f}" if val['jrn'] else "N/A",
                 f"{val['sensitivity']:.4f}"])
print(tabulate(rows, headers=["Task/Join", "JRN", "Sensitivity"], tablefmt="simple"))

# Category distribution
print(f"\nJRN Category Distribution:")
print("-" * 60)
cd = analysis["category_distribution"]
cd_rows = []
for tn, counts in cd.items():
    cd_rows.append([tn, counts.get("HIGH", 0), counts.get("CRITICAL", 0),
                    counts.get("LOW", 0), counts.get("N/A", 0)])
print(tabulate(cd_rows, headers=["Task", "HIGH", "CRITICAL", "LOW", "N/A"],
               tablefmt="simple"))

print("\n" + "=" * 60)
print("Demo complete! All 11 joins classified as HIGH or CRITICAL,")
print("confirming all FK relationships are informative for rel-stack.")
print("Oracle gaps are small (0.9%-8.2%), showing JRN-guided is near-optimal.")
print("=" * 60)